In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Carregando os datasets em um dicionário
# CEP é lido como string: como inteiro os zeros à esquerda se perdem (ex.: 01046 -> 1046)
zip_dtypes = {
    "customer_zip_code_prefix": str,
    "seller_zip_code_prefix": str,
    "geolocation_zip_code_prefix": str,
}

datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file, dtype=zip_dtypes)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [4]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [5]:
# Carregando dataframes
sellers_df = pd.DataFrame(datasets["olist_sellers_dataset"])
order_items_df = pd.DataFrame(datasets["olist_order_items_dataset"])
geolocation_df = pd.DataFrame(datasets["olist_geolocation_dataset"])
sellers_df.shape

(3095, 4)

In [6]:
sellers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   str  
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: str(4)
memory usage: 96.8 KB


In [7]:
#Avaliando se existe valores nulos
sellers_df.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [8]:
#Avaliando se existe dados duplicados
print(f"Linhas totalmente duplicadas: {sellers_df.duplicated().sum()}")
print(f"seller_id duplicados: {sellers_df['seller_id'].duplicated().sum()}")

Linhas totalmente duplicadas: 0
seller_id duplicados: 0


In [9]:
# CEP: precisa ter 5 dígitos (string preserva os zeros à esquerda)
print(sellers_df['seller_zip_code_prefix'].dtype)
print(sellers_df['seller_zip_code_prefix'].str.len().value_counts())

str
seller_zip_code_prefix
5    3095
Name: count, dtype: int64


In [10]:
UFS = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA",
       "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}

print(f"Estados distintos: {sellers_df['seller_state'].nunique()}")
print(f"Estados inválidos: {(~sellers_df['seller_state'].isin(UFS)).sum()}")
sellers_df['seller_state'].value_counts().head(10)

Estados distintos: 23
Estados inválidos: 0


seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64

In [11]:
# Cidades com valores suspeitos (barras, e-mail, números, espaços nas pontas)
dirty_pattern = r"[0-9@\\/]|^\s|\s$"
dirty_mask = sellers_df['seller_city'].str.contains(dirty_pattern)

print(f"Cidades distintas: {sellers_df['seller_city'].nunique()}")
print(f"Registros com cidade suspeita: {dirty_mask.sum()}")
sellers_df.loc[dirty_mask, ['seller_id', 'seller_city', 'seller_zip_code_prefix', 'seller_state']]

Cidades distintas: 611
Registros com cidade suspeita: 18


,seller_id,seller_city,seller_zip_code_prefix,seller_state
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,15350,SP
246,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,03407,SP
517,ceb7b4fb9401cd378de7886317ad1b47,04482255,22790,RJ
622,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,29142,ES
869,cbf09e831b0c11f6f23ffb51004db972,sbc/sp,09726,SP
945,f52c2422904463fdd7741f99045fecb6,santo andre/sao paulo,09230,SP
1004,1cbd32d00d01bb8087a5eb088612fd9c,sp / sp,03363,SP
1159,89dda63a3c907c468ec88c310ed91213,maua/sao paulo,09380,SP
1337,720e6cf846ea7572cbb66b743fb91e6c,mogi das cruzes / sp,08717,SP
1346,cf1313c6e2c01c2f4b014f97db4bcd2b,rio de janeiro \rio de janeiro,22050,RJ


In [12]:
# Função para padronizar texto: sem acento, minúsculo e sem espaços nas pontas
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return text.lower().strip()

In [13]:
# Cidade mais frequente na geolocation para cada CEP (referência para corrigir as cidades suspeitas)
geo_cities = geolocation_df[['geolocation_zip_code_prefix', 'geolocation_city']].copy()
geo_cities['geolocation_city'] = geo_cities['geolocation_city'].map(normalize_text)

geo_city_by_zip = geo_cities.groupby('geolocation_zip_code_prefix')['geolocation_city'].agg(lambda s: s.mode().iat[0])
geo_city_by_zip.head()

geolocation_zip_code_prefix
01001    sao paulo
01002    sao paulo
01003    sao paulo
01004    sao paulo
01005    sao paulo
Name: geolocation_city, dtype: str

In [14]:
# Comparando a cidade suspeita com a cidade da geolocation para o mesmo CEP
check = sellers_df.loc[dirty_mask, ['seller_city', 'seller_zip_code_prefix']].copy()
check['geo_city'] = check['seller_zip_code_prefix'].map(geo_city_by_zip)

print(f"Cidades suspeitas sem CEP na geolocation: {check['geo_city'].isna().sum()}")
check

Cidades suspeitas sem CEP na geolocation: 0


,seller_city,seller_zip_code_prefix,geo_city
237,auriflama/sp,15350,auriflama
246,sao paulo / sao paulo,03407,sao paulo
517,04482255,22790,rio de janeiro
622,cariacica / es,29142,cariacica
869,sbc/sp,09726,sao bernardo do campo
945,santo andre/sao paulo,09230,santo andre
1004,sp / sp,03363,sao paulo
1159,maua/sao paulo,09380,maua
1337,mogi das cruzes / sp,08717,mogi das cruzes
1346,rio de janeiro \rio de janeiro,22050,rio de janeiro


In [15]:
# Integridade com order_items
print(f"seller_id de order_items sem correspondência em sellers: {(~order_items_df['seller_id'].isin(sellers_df['seller_id'])).sum()}")
print(f"Vendedores sem nenhuma venda: {(~sellers_df['seller_id'].isin(order_items_df['seller_id'])).sum()}")

missing_zip = ~sellers_df['seller_zip_code_prefix'].isin(geolocation_df['geolocation_zip_code_prefix'])
print(f"Vendedores com CEP sem correspondência na geolocation: {missing_zip.sum()}")

seller_id de order_items sem correspondência em sellers: 0
Vendedores sem nenhuma venda: 0
Vendedores com CEP sem correspondência na geolocation: 7


In [16]:
#Fazendo copia do dataframe para não alterar o original
sellers_clean = sellers_df.copy()

In [17]:
# Padronizando texto
sellers_clean['seller_city'] = sellers_clean['seller_city'].map(normalize_text)
sellers_clean['seller_state'] = sellers_clean['seller_state'].str.strip().str.upper()
sellers_clean['seller_zip_code_prefix'] = sellers_clean['seller_zip_code_prefix'].str.strip().str.zfill(5)

# Cidades suspeitas passam a usar a cidade da geolocation para o mesmo CEP
sellers_clean.loc[dirty_mask, 'seller_city'] = sellers_clean.loc[dirty_mask, 'seller_zip_code_prefix'].map(geo_city_by_zip)

In [18]:
# Resultado da correção das cidades suspeitas
pd.DataFrame({
    "antes": sellers_df.loc[dirty_mask, 'seller_city'],
    "depois": sellers_clean.loc[dirty_mask, 'seller_city'],
})

,antes,depois
237,auriflama/sp,auriflama
246,sao paulo / sao paulo,sao paulo
517,04482255,rio de janeiro
622,cariacica / es,cariacica
869,sbc/sp,sao bernardo do campo
945,santo andre/sao paulo,santo andre
1004,sp / sp,sao paulo
1159,maua/sao paulo,maua
1337,mogi das cruzes / sp,mogi das cruzes
1346,rio de janeiro \rio de janeiro,rio de janeiro


In [19]:
# Verificando quantos valores foram alterados em cada coluna
for col in sellers_df.columns:
    print(f"{col}: {(sellers_df[col] != sellers_clean[col]).sum()} valores alterados")

seller_id: 0 valores alterados
seller_zip_code_prefix: 0 valores alterados
seller_city: 21 valores alterados
seller_state: 0 valores alterados


In [20]:
# Validações pós-tratamento
print(f"Linhas: {sellers_clean.shape[0]} (original: {sellers_df.shape[0]})")
print(f"Valores nulos: {sellers_clean.isna().sum().sum()}")
print(f"seller_id duplicados: {sellers_clean['seller_id'].duplicated().sum()}")
print(f"Cidades ainda suspeitas: {sellers_clean['seller_city'].str.contains(dirty_pattern).sum()}")
print(f"CEPs com tamanho diferente de 5: {(sellers_clean['seller_zip_code_prefix'].str.len() != 5).sum()}")
print(f"Estados inválidos: {(~sellers_clean['seller_state'].isin(UFS)).sum()}")
print(f"seller_id de order_items sem correspondência: {(~order_items_df['seller_id'].isin(sellers_clean['seller_id'])).sum()}")

assert sellers_clean.shape == sellers_df.shape
assert sellers_clean['seller_id'].is_unique
assert not sellers_clean['seller_city'].str.contains(dirty_pattern).any()

Linhas: 3095 (original: 3095)
Valores nulos: 0
seller_id duplicados: 0
Cidades ainda suspeitas: 0
CEPs com tamanho diferente de 5: 0
Estados inválidos: 0
seller_id de order_items sem correspondência: 0


In [21]:
# Salvando dataframe tratado em pasta processed
sellers_clean.to_csv('../data/processed/olist_sellers_dataset_clean.csv', index=False)

## 📌 Conclusão da etapa — Sellers

### 🔎 O que foi encontrado?

A tabela `sellers` possui **3.095 registros e 4 colunas**.

Durante o profiling, foram identificados:

- Nenhum valor nulo e nenhuma duplicidade (`seller_id` é único);
- Vendedores em **23 estados**, todos UFs válidas, e todos os CEPs com 5 dígitos;
- **18 registros com `seller_city` sujo**: UF ou estado junto da cidade (`auriflama/sp`, `sao paulo / sao paulo`), abreviação (`sbc/sp`), CEP no lugar da cidade (`04482255`) e até um e-mail (`vendas@creditparts.com.br`);
- **7 vendedores** com CEP que não existe em `geolocation`;
- Todos os vendedores possuem vendas e todo `seller_id` de `order_items` existe em `sellers`.

### 🛠️ O que foi feito?

- Cidade padronizada para minúsculo, sem acento e sem espaços nas pontas (mesmo padrão de `customers`);
- Para os 18 registros sujos, a cidade foi substituída pela **cidade mais frequente em `geolocation` para o mesmo CEP**. Todos os 18 CEPs existiam em `geolocation` e as correções conferem com a cidade original (ex.: `sbc/sp` → `sao bernardo do campo`, `04482255` → `rio de janeiro`, e-mail → `maringa`);
- CEP mantido como string com 5 dígitos.

No total, **21 valores de `seller_city`** foram alterados (18 corrigidos + 3 de padronização). Nenhum registro foi removido.

### ✅ Como foi validado?

- A quantidade de registros permaneceu em **3.095**;
- Sem nulos, sem `seller_id` duplicado e sem cidades suspeitas;
- Estados e CEPs válidos;
- Todos os `seller_id` de `order_items` possuem correspondência em `sellers`.

### 📁 Resultado

O dataset tratado foi salvo em:

`data/processed/olist_sellers_dataset_clean.csv`